In [2]:
import os, time, re, warnings, copy
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import find_peaks

from openpyxl import load_workbook, Workbook
from openpyxl.styles import Alignment, Border, Side, Font, PatternFill
from openpyxl.utils import get_column_letter
from paddleocr import PaddleOCR

# ── PATHS ─────────────────────────────────────────────────────────
INPUT_FOLDER  = 'data/images'
OUTPUT_FOLDER = 'data/output'
TEMPLATE_PATH = 'data/template/template.xlsx'
ACTUAL_FOLDER = 'data/actual'

IMG_NAME  = 'IMG_1703.jpeg'
img_path  = os.path.join(INPUT_FOLDER, IMG_NAME)
base      = Path(img_path).stem
out_excel = os.path.join(OUTPUT_FOLDER, f'{base}_filled.xlsx')
out_acc   = os.path.join(OUTPUT_FOLDER, f'{base}_accuracy.csv')
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

SHOW_DEBUG = False

# ── OCR & CELL PARAMS ─────────────────────────────────────────────
OCR_MIN_CONF      = 0.20
CELL_TARGET_H_PX  = 120
CELL_MAX_SCALE    = 8.0
CELL_BORDER_STRIP = 3
CELL_PAD_PX       = 8

NARROW_COLS        = {9, 10, 11, 12, 13}
NARROW_TARGET_H_PX = 160
NARROW_MAX_SCALE   = 12.0

# ── BLEED-THROUGH FILTER ──────────────────────────────────────────
BLEED_CONF_THRESHOLD = 0.32
BLEED_PATTERNS = [
    r'^[^a-zA-Z0-9]+$',
    r'^[.\-_,;:!?]{1,3}$',
    r'^.{1,2}$',
    r'[\\|/]{2,}',
]

# ── COLUMN TYPES ──────────────────────────────────────────────────
NUMERIC_COLS      = {0, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16}
TEXT_COLS         = {1, 5}
MIN_COLS_EXPECTED = 17
# How many rows remain after skipping headers
# Formula: skip = nrows - EXPECTED_ROWS_AFTER_SKIP
# Grid 69 -> skip=4 | Grid 70 -> skip=5 | Grid 71 -> skip=6
EXPECTED_ROWS_AFTER_SKIP = 65

TEMPLATE_HEADER_ROWS = 6
TEMPLATE_TOTAL_LABEL = 'TOTAL'

# ── DASH FILTER ───────────────────────────────────────────────────
DASH_ONLY = re.compile(r'^[\s.\-_,]+$')

# ── EXCEL STYLES ──────────────────────────────────────────────────
_thin  = Side(style='thin',   color='000000')
_thick = Side(style='medium', color='000000')
BORDER_THIN  = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)
FILL_DATA    = PatternFill('solid', fgColor='FFFDE7')
FILL_TOTAL   = PatternFill('solid', fgColor='BBDEFB')
FILL_LOWCONF = PatternFill('solid', fgColor='FFF2CC')
def px_to_col_width(px): return max(1.0, (px-5)/7.0)
def px_to_row_ht(px):    return max(5.0, px*0.75)

print('Config v4.9 ready.')


Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


Config v4.9 ready.


In [3]:
# ══════════════════════════════════════════════════════════════════
# SECTION FUZZY – MULTI-METHOD FUZZY STRING MATCHER
# ══════════════════════════════════════════════════════════════════
# Methods combined:
#   1. Levenshtein distance        — counts insert/delete/substitute
#   2. Weighted edit distance      — OCR-specific weights (B↔S, 0↔O, etc)
#   3. Jaro-Winkler similarity     — prefix bonus (good for month names)
#   4. Bigram (q-gram) index       — overlapping 2-char set similarity
#   5. Permterm index              — sorted character-set overlap
#
# Each method returns 0.0-1.0 similarity. Combined as weighted vote.
# ══════════════════════════════════════════════════════════════════

# ── 1. Levenshtein Distance ───────────────────────────────────────
def levenshtein(s1, s2):
    """Standard edit distance. Returns similarity in [0,1]."""
    s1, s2 = s1.lower(), s2.lower()
    if s1 == s2: return 1.0
    if not s1 or not s2: return 0.0
    m, n = len(s1), len(s2)
    dp = list(range(n+1))
    for i in range(1, m+1):
        prev, dp[0] = dp[0], i
        for j in range(1, n+1):
            prev, dp[j] = dp[j], (prev if s1[i-1]==s2[j-1]
                                   else 1 + min(prev, dp[j], dp[j-1]))
    dist = dp[n]
    return 1.0 - dist / max(m, n)

# ── 2. Weighted Edit Distance ─────────────────────────────────────
# OCR confusion matrix: pairs that look similar in handwriting/print
OCR_CONFUSION = {
    # (char1, char2): substitution_cost  (lower = more similar)
    ('b','s'):0.1, ('s','b'):0.1,   # Bep → Sep
    ('0','o'):0.1, ('o','0'):0.1,   # 017 → Oct
    ('1','l'):0.2, ('l','1'):0.2,
    ('1','i'):0.2, ('i','1'):0.2,
    ('u','n'):0.2, ('n','u'):0.2,
    ('c','e'):0.2, ('e','c'):0.2,
    ('a','u'):0.3, ('u','a'):0.3,   # Jau → Jan
    ('v','y'):0.2, ('y','v'):0.2,
    ('m','n'):0.2, ('n','m'):0.2,
    ('d','a'):0.3, ('a','d'):0.3,
    ('f','t'):0.3, ('t','f'):0.3,
    ('p','r'):0.2, ('r','p'):0.2,
    ('g','q'):0.1, ('q','g'):0.1,
    ('6','b'):0.1, ('b','6'):0.1,
    ('4','a'):0.3, ('a','4'):0.3,
}

def weighted_edit_distance(s1, s2):
    """Edit distance with OCR-aware substitution costs. Returns [0,1] sim."""
    s1, s2 = s1.lower(), s2.lower()
    if s1 == s2: return 1.0
    if not s1 or not s2: return 0.0
    m, n = len(s1), len(s2)
    dp = [[0.0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = float(i)
    for j in range(n+1): dp[0][j] = float(j)
    for i in range(1, m+1):
        for j in range(1, n+1):
            if s1[i-1] == s2[j-1]:
                cost = 0.0
            else:
                cost = OCR_CONFUSION.get((s1[i-1], s2[j-1]), 1.0)
            dp[i][j] = min(
                dp[i-1][j] + 1.0,      # delete
                dp[i][j-1] + 1.0,      # insert
                dp[i-1][j-1] + cost    # substitute (OCR-aware)
            )
    # Transposition (Damerau): swapped adjacent chars
    if i > 1 and j > 1 and s1[i-1]==s2[j-2] and s1[i-2]==s2[j-1]:
        dp[i][j] = min(dp[i][j], dp[i-2][j-2] + 0.5)
    max_cost = max(m, n)
    return max(0.0, 1.0 - dp[m][n] / max_cost)

# ── 3. Jaro-Winkler Similarity ────────────────────────────────────
def jaro_winkler(s1, s2, p=0.1, max_prefix=4):
    """
    Jaro-Winkler: rewards common prefix.
    Best for month names (Apr/Ap/Apu all share 'a' prefix).
    """
    s1, s2 = s1.lower(), s2.lower()
    if s1 == s2: return 1.0
    if not s1 or not s2: return 0.0
    m1, m2 = len(s1), len(s2)
    match_dist = max(m1, m2) // 2 - 1
    if match_dist < 0: match_dist = 0
    s1_matches = [False]*m1; s2_matches = [False]*m2
    matches = 0; transpositions = 0
    for i in range(m1):
        lo = max(0, i-match_dist)
        hi = min(i+match_dist+1, m2)
        for j in range(lo, hi):
            if s2_matches[j] or s1[i] != s2[j]: continue
            s1_matches[i] = s2_matches[j] = True
            matches += 1; break
    if matches == 0: return 0.0
    k = 0
    for i in range(m1):
        if not s1_matches[i]: continue
        while not s2_matches[k]: k += 1
        if s1[i] != s2[k]: transpositions += 1
        k += 1
    jaro = (matches/m1 + matches/m2 +
            (matches - transpositions/2)/matches) / 3
    prefix = 0
    for i in range(min(max_prefix, m1, m2)):
        if s1[i] == s2[i]: prefix += 1
        else: break
    return jaro + prefix * p * (1 - jaro)

# ── 4. Q-gram (Bigram) Index Similarity ──────────────────────────
def qgram_similarity(s1, s2, q=2):
    """
    Overlapping q-gram similarity.
    'sep' bigrams: {se, ep}  vs  'bep' bigrams: {be, ep}
    Overlap = {ep} → sim = 2*|overlap|/(|q1|+|q2|)
    """
    s1, s2 = s1.lower(), s2.lower()
    if s1 == s2: return 1.0
    if len(s1)<q or len(s2)<q: return float(s1==s2)
    def grams(s): return [s[i:i+q] for i in range(len(s)-q+1)]
    g1, g2 = grams(s1), grams(s2)
    # Use multiset intersection
    from collections import Counter
    c1, c2 = Counter(g1), Counter(g2)
    inter = sum((c1 & c2).values())
    return 2.0 * inter / (len(g1) + len(g2)) if (len(g1)+len(g2)) > 0 else 0.0

# ── 5. Permterm Index Similarity ─────────────────────────────────
def permterm_similarity(s1, s2):
    """
    Character-set (permterm) similarity: sorted chars overlap.
    'bep' → {b,e,p}  vs  'sep' → {e,p,s}
    Overlap = {e,p}/union = 2/4 = 0.5
    Good for detecting anagram-like OCR confusions.
    """
    s1, s2 = s1.lower(), s2.lower()
    if s1 == s2: return 1.0
    from collections import Counter
    c1, c2 = Counter(s1), Counter(s2)
    inter = sum((c1 & c2).values())
    union = sum((c1 | c2).values())
    return inter / union if union > 0 else 0.0

# ── COMBINED FUZZY SCORE ──────────────────────────────────────────
# Weights for each method (tuned for handwriting OCR context)
FUZZY_WEIGHTS = {
    'levenshtein':    0.25,   # solid baseline
    'weighted_edit':  0.35,   # most important: OCR-aware costs
    'jaro_winkler':   0.20,   # prefix match bonus
    'qgram':          0.12,   # bigram overlap
    'permterm':       0.08,   # char-set overlap
}

def fuzzy_score(ocr_text, candidate):
    """Combined fuzzy score ∈ [0,1] between OCR string and candidate."""
    s = (FUZZY_WEIGHTS['levenshtein']   * levenshtein(ocr_text, candidate) +
         FUZZY_WEIGHTS['weighted_edit'] * weighted_edit_distance(ocr_text, candidate) +
         FUZZY_WEIGHTS['jaro_winkler']  * jaro_winkler(ocr_text, candidate) +
         FUZZY_WEIGHTS['qgram']         * qgram_similarity(ocr_text, candidate) +
         FUZZY_WEIGHTS['permterm']      * permterm_similarity(ocr_text, candidate))
    return s

def fuzzy_best_match(ocr_text, vocabulary, min_score=0.40):
    """
    Find best match for ocr_text in vocabulary using combined fuzzy score.
    Returns (best_match, score) or (None, 0.0) if below min_score.
    """
    if not ocr_text: return None, 0.0
    t = ocr_text.strip().lower()
    scores = [(cand, fuzzy_score(t, cand.lower())) for cand in vocabulary]
    scores.sort(key=lambda x: -x[1])
    best_cand, best_sc = scores[0]
    if best_sc >= min_score:
        return best_cand, best_sc
    return None, 0.0

# ── MONTH VOCABULARY ──────────────────────────────────────────────
MONTH_VOCAB = [
    'Jan','Feb','Mar','Apr','May','Jun',
    'Jul','Aug','Sep','Oct','Nov','Dec',
    'PR','PRN'   # pruning indicators
]
# Short aliases included for matching
MONTH_VOCAB_EXTENDED = MONTH_VOCAB + [
    'January','February','March','April','May','June',
    'July','August','September','October','November','December',
]
# Canonical mapping: any month variant → standard 3-letter
MONTH_CANONICAL = {
    'January':'Jan','February':'Feb','March':'Mar','April':'Apr',
    'May':'May','June':'Jun','July':'Jul','August':'Aug',
    'September':'Sep','October':'Oct','November':'Nov','December':'Dec',
    'Jan':'Jan','Feb':'Feb','Mar':'Mar','Apr':'Apr',
    'Jun':'Jun','Jul':'Jul','Aug':'Aug','Sep':'Sep',
    'Oct':'Oct','Nov':'Nov','Dec':'Dec','PR':'PR','PRN':'PRN',
}

# ── FERTILIZER VOCABULARY ─────────────────────────────────────────
FERT_VOCAB = [
    'U-709','U-330','U-428','T-750',
    'B/A','B/N','DAP','MOP',
    'PH-B','PH-4.99A','PH-5.24A',
]
# Numeric suffix patterns (U-NNN) extracted separately
FERT_NUM_PAT = re.compile(
    r'\b([Uu][_\-]?|[Tt][_\-]?)(\d{2,4})\b')
FERT_KNOWN_NUMS = {'709':'U-709','330':'U-330','428':'U-428','750':'T-750'}

def fuzzy_match_month(ocr_text):
    """
    Match OCR month string to standard month using combined fuzzy scoring.
    Steps:
      1. Direct exact lookup (fastest)
      2. Prefix match (3-char)
      3. Full fuzzy match against extended vocabulary
    Returns standard month string or original if no good match.
    """
    t = str(ocr_text).strip()
    if not t: return t
    tl = t.lower()

    # Step 1: Direct exact (covers common correct reads)
    for m in MONTH_VOCAB_EXTENDED:
        if tl == m.lower():
            return MONTH_CANONICAL.get(m, m[:3].title())

    # Step 2: 3-char prefix → fast lookup
    if len(tl) >= 3:
        for m in MONTH_VOCAB_EXTENDED:
            if m.lower().startswith(tl[:3]) or tl[:3] == m.lower()[:3]:
                return MONTH_CANONICAL.get(m, m[:3].title())

    # Step 3: Fuzzy match against all month names
    best, sc = fuzzy_best_match(tl, MONTH_VOCAB_EXTENDED, min_score=0.42)
    if best:
        canonical = MONTH_CANONICAL.get(best, best[:3].title())
        return canonical

    # Keep original if no match found
    return t[:3].title() if t[:3].isalpha() else t

def fuzzy_match_fert(ocr_text):
    """
    Match OCR fertilizer code using fuzzy scoring.
    First tries to extract numeric suffix (U-709 pattern),
    then falls back to fuzzy match against vocabulary.
    """
    t = str(ocr_text).strip()
    if not t: return t

    # Step 1: Extract U-NNN / T-NNN pattern directly
    m = FERT_NUM_PAT.search(t)
    if m:
        num = m.group(2)
        prefix = m.group(1).upper().rstrip('-_')
        if num in FERT_KNOWN_NUMS:
            return FERT_KNOWN_NUMS[num]
        return f"{prefix}-{num}"

    # Step 2: OCR-garbled U-codes (e.g. 4709→U-709, tanl→U-330)
    # Extract 3-4 digit runs that look like fertilizer numbers
    digits = re.findall(r'\d{3,4}', t)
    for d in digits:
        if d in FERT_KNOWN_NUMS:
            return FERT_KNOWN_NUMS[d]

    # Step 3: Fuzzy match against vocabulary
    best, sc = fuzzy_best_match(t, FERT_VOCAB, min_score=0.38)
    if best:
        return best

    # Step 4: B/A, DAP, MOP pattern matching
    tl = t.lower()
    if re.search(r'b[/\\]?a', tl): return 'B/A'
    if re.search(r'b[/\\]?n', tl): return 'B/N'
    if 'dap' in tl: return 'DAP'
    if 'mop' in tl: return 'MOP'
    if re.search(r'ph[-\s]?b', tl): return 'PH-B'

    return ' '.join(t.split())

# ── DEMO / TEST ───────────────────────────────────────────────────
print('Fuzzy Matcher ready. Testing...')
print()
test_months = ['Bep','Nnv','Auu','Jau','Fcd','017','Col','Mav',
               'Tun','Sct','Aot','Nec','Fob','Map','Seo','T.h']
print('Month Corrections:')
for m in test_months:
    result = fuzzy_match_month(m)
    print(f'  {m:<8} → {result}')

print()
test_ferts = ['4ES30','4709','4428','tanl','UFO9','47o9','0709',
              '1-209','T750','4709 80','B/A']
print('Fertilizer Corrections:')
for f in test_ferts:
    result = fuzzy_match_fert(f)
    print(f'  {f:<10} → {result}')


Fuzzy Matcher ready. Testing...

Month Corrections:
  Bep      → Sep
  Nnv      → Nov
  Auu      → Aug
  Jau      → Jan
  Fcd      → Feb
  017      → 017
  Col      → Col
  Mav      → May
  Tun      → Jun
  Sct      → Oct
  Aot      → Aot
  Nec      → Dec
  Fob      → Feb
  Map      → Mar
  Seo      → Sep
  T.h      → T.h

Fertilizer Corrections:
  4ES30      → U-330
  4709       → U-709
  4428       → U-428
  tanl       → tanl
  UFO9       → U-709
  47o9       → U-709
  0709       → U-709
  1-209      → U-709
  T750       → T-750
  4709 80    → U-709
  B/A        → B/A


In [4]:
# ══════════════════════════════════════════════════════════════════
# SECTION A – OCR ENGINE  (unchanged from previous working code)
# ══════════════════════════════════════════════════════════════════
def make_ocr_engine():
    for kwargs in [
        dict(lang='en', det_model_name='PP-OCRv5_server_det',
             rec_model_name='PP-OCRv5_server_rec', use_textline_orientation=False),
        dict(lang='en', ocr_version='PP-OCRv5', use_textline_orientation=False),
        dict(lang='en', use_textline_orientation=False),
        dict(lang='en'),
    ]:
        try:
            eng = PaddleOCR(**kwargs)
            print(f'  OCR loaded: {list(kwargs.keys())}')
            return eng
        except Exception:
            continue
    raise RuntimeError('PaddleOCR init failed')

OCR_ENGINE = make_ocr_engine()

def _parse_paddle_output(out):
    items = []
    if not out: return items
    if isinstance(out, list) and len(out) > 0 and isinstance(out[0], dict):
        d = out[0]
        texts  = d.get('rec_texts') or d.get('texts') or []
        scores = d.get('rec_scores') or d.get('scores') or ([1.0]*len(texts))
        boxes  = (d.get('dt_polys') or d.get('rec_polys') or
                  d.get('dt_boxes') or d.get('boxes') or [])
        if isinstance(boxes, np.ndarray): boxes = boxes.tolist()
        if isinstance(texts, str): texts = [texts]
        for b,t,s in zip(boxes, texts, scores):
            items.append((b, t, float(s)))
        return items
    lines = out[0] if (isinstance(out[0], list) and
                       not isinstance(out[0][0], list)) else out
    for line in lines:
        try: items.append((line[0], line[1][0], float(line[1][1])))
        except Exception: pass
    return items

def run_ocr(img_bgr):
    if img_bgr is None or img_bgr.size == 0: return []
    if img_bgr.ndim == 2:
        img_bgr = cv2.cvtColor(img_bgr, cv2.COLOR_GRAY2BGR)
    try:
        if hasattr(OCR_ENGINE, 'predict'):
            try:
                out = OCR_ENGINE.predict(img_bgr,
                    use_doc_orientation_classify=False,
                    use_doc_unwarping=False,
                    use_textline_orientation=False)
            except TypeError:
                out = OCR_ENGINE.predict(img_bgr)
        else:
            out = OCR_ENGINE.ocr(img_bgr)
        return _parse_paddle_output(out)
    except Exception:
        return []

print('OCR engine ready.')


Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\User\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\User\.paddlex\official_models\UVDoc`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\User\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\User\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.


  OCR loaded: ['lang', 'ocr_version', 'use_textline_orientation']
OCR engine ready.


In [5]:
# ══════════════════════════════════════════════════════════════════
# SECTION B – IMAGE + GRID  (previous code + enforce_min_columns)
# ══════════════════════════════════════════════════════════════════
def _four_point_warp(bgr, pts):
    pts = pts.reshape(4,2); s = pts.sum(1); d = np.diff(pts, axis=1)
    rect = np.array([pts[s.argmin()], pts[d.argmin()],
                     pts[s.argmax()], pts[d.argmax()]], dtype=np.float32)
    tl,tr,br,bl = rect
    W = int(max(np.linalg.norm(br-bl), np.linalg.norm(tr-tl)))
    H = int(max(np.linalg.norm(tr-br), np.linalg.norm(tl-bl)))
    dst = np.array([[0,0],[W-1,0],[W-1,H-1],[0,H-1]], dtype=np.float32)
    return cv2.warpPerspective(bgr, cv2.getPerspectiveTransform(rect,dst), (W,H))

def rectify_document(bgr):
    gray  = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray,(5,5),0), 50, 150)
    edges = cv2.dilate(edges, np.ones((5,5), np.uint8))
    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return bgr
    for c in sorted(cnts, key=cv2.contourArea, reverse=True)[:5]:
        ap = cv2.approxPolyDP(c, 0.02*cv2.arcLength(c,True), True)
        if len(ap) == 4:
            return _four_point_warp(bgr, ap)
    return bgr

def split_left_right(bgr, pad=3):
    h, w = bgr.shape[:2]
    _, bi = cv2.threshold(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY),
                          200, 255, cv2.THRESH_BINARY_INV)
    vk = max(30, h//18)
    vm = cv2.morphologyEx(bi, cv2.MORPH_OPEN,
                          cv2.getStructuringElement(cv2.MORPH_RECT,(1,vk)))
    proj = np.convolve(
        np.sum(vm[int(h*0.2):, :] > 0, axis=0).astype(float),
        np.ones(max(15,w//200))/max(15,w//200), mode='same')
    if proj.max() == 0: return [(1, bgr)]
    peaks, _ = find_peaks(proj, height=proj.max()*0.25,
                          distance=max(12,w//100))
    if len(peaks) == 0: return [(1, bgr)]
    mid = peaks[np.argmin(np.abs(peaks - w*0.5))]
    tables = [(i, bgr[:, max(0,x1-pad):min(w,x2+pad)].copy())
              for i,(x1,x2) in enumerate([(0,mid),(mid,w)],1)
              if x2-x1 > w*0.1]
    return tables if len(tables)==2 else [(1, bgr)]

def _cluster(pos, eps=8):
    pos = np.sort(np.unique(pos.astype(int)))
    if len(pos) == 0: return pos
    groups, cur = [[pos[0]]], pos[0]
    for p in pos[1:]:
        if p-cur <= eps: groups[-1].append(p)
        else: groups.append([p])
        cur = p
    return np.array([int(np.mean(g)) for g in groups])

def detect_grid(bgr):
    h, w = bgr.shape[:2]
    g  = cv2.createCLAHE(2.0,(8,8)).apply(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY))
    th = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY_INV, 31, 15)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, np.ones((3,3),np.uint8), iterations=2)
    hk = max(20,w//30); vk = max(20,h//30)
    hl = cv2.dilate(cv2.morphologyEx(th, cv2.MORPH_OPEN,
                    cv2.getStructuringElement(cv2.MORPH_RECT,(hk,1))),
                    np.ones((1,3),np.uint8))
    vl = cv2.dilate(cv2.morphologyEx(th, cv2.MORPH_OPEN,
                    cv2.getStructuringElement(cv2.MORPH_RECT,(1,vk))),
                    np.ones((3,1),np.uint8))
    ys = [y+hh//2 for c in cv2.findContours(hl,cv2.RETR_EXTERNAL,
          cv2.CHAIN_APPROX_SIMPLE)[0]
          for x,y,ww,hh in [cv2.boundingRect(c)] if ww >= w*0.40]
    xs = [x+ww//2 for c in cv2.findContours(vl,cv2.RETR_EXTERNAL,
          cv2.CHAIN_APPROX_SIMPLE)[0]
          for x,y,ww,hh in [cv2.boundingRect(c)] if hh >= h*0.40]
    ys = _cluster(np.array(ys)); xs = _cluster(np.array(xs))
    def _border(lines, lo, hi, tol=14):
        l = list(lines)
        if not l or abs(l[0]-lo) > tol: l = [lo]+l
        if abs(l[-1]-hi) > tol: l += [hi]
        return np.array(sorted(l))
    return _border(ys,0,h-1), _border(xs,0,w-1)

def enforce_min_columns(vlines, min_cols=MIN_COLS_EXPECTED):
    """Split the widest column until we have at least min_cols."""
    vl = list(vlines)
    while len(vl)-1 < min_cols:
        widths = [(vl[i+1]-vl[i], i) for i in range(len(vl)-1)]
        _, idx = max(widths)
        mid = (vl[idx]+vl[idx+1])//2
        vl.insert(idx+1, mid)
    return np.array(vl, dtype=float)


# ── AUTO TABLE LAYOUT DETECTION (2-table vs 3-table) ──────────────
def split_three_tables(bgr, pad=3):
    """
    Detect and split a 3-table page layout using zonal gap analysis.
    Scans 3 zones on the vertical-line projection to find the two
    separator gaps that divide Left / Center / Right tables.
    Returns [(1,img1),(2,img2),(3,img3)]
    """
    h, w = bgr.shape[:2]
    _, bi = cv2.threshold(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY),
                          200, 255, cv2.THRESH_BINARY_INV)
    vk = max(int(h / 20), 30)
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, vk))
    vert_clean = cv2.morphologyEx(bi, cv2.MORPH_OPEN, v_kernel)
    proj = np.sum(vert_clean, axis=0).astype(float)
    if proj.max() == 0:
        return [(1, bgr)]
    from scipy.signal import find_peaks as _fp
    peaks, _ = _fp(proj, height=proj.max() * 0.10, distance=5)

    zones = {
        'A': (int(w * 0.25), int(w * 0.40)),
        'B': (int(w * 0.45), int(w * 0.55)),
        'C': (int(w * 0.60), int(w * 0.75)),
    }
    noise_floor = w * 0.015
    best = {}
    for zn, (zx0, zx1) in zones.items():
        zp = [p for p in peaks if zx0 <= p <= zx1]
        if len(zp) < 2:
            best[zn] = {'size': 0, 'cut_x': 0}
            continue
        gaps = np.diff(zp)
        idx  = int(np.argmax(gaps))
        best[zn] = {'size': int(gaps[idx]), 'cut_x': int(zp[idx + 1])}

    cuts = []
    if best['A']['size'] > noise_floor and best['A']['size'] >= best['B']['size']:
        cuts.append(best['A']['cut_x'])
        if best['C']['size'] > noise_floor:
            cuts.append(best['C']['cut_x'])
        else:
            cuts.append(int(w * 0.66))
    else:
        cx = best['B']['cut_x'] if best['B']['size'] > noise_floor else int(w * 0.50)
        cuts.append(cx)

    boundaries = [0] + sorted(cuts) + [w]
    tables = []
    for i in range(len(boundaries) - 1):
        x1, x2 = boundaries[i], boundaries[i + 1]
        if (x2 - x1) > w * 0.10:
            tables.append((i + 1, bgr[:, max(0, x1 - pad):min(w, x2 + pad)].copy()))
    return tables if tables else [(1, bgr)]


def auto_split_tables(bgr, pad=3):
    """
    Auto-detect whether the page has 2 tables or 3 tables.

    Detection method:
      - Build vertical-line projection
      - Check for strong gaps in Zone A (25-40 %) and Zone B (45-55 %)
      - If Zone A gap > Zone B gap  → 3-table layout
      - Otherwise                   → 2-table layout

    Returns:
        layout  : '2-table' or '3-table'
        tables  : [(t_idx, img_bgr), ...]
    """
    h, w = bgr.shape[:2]
    _, bi = cv2.threshold(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY),
                          200, 255, cv2.THRESH_BINARY_INV)
    vk = max(int(h / 20), 30)
    vert_clean = cv2.morphologyEx(
        bi, cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (1, vk)))
    proj = np.sum(vert_clean, axis=0).astype(float)
    if proj.max() == 0:
        return '2-table', [(1, bgr)]

    from scipy.signal import find_peaks as _fp
    peaks, _ = _fp(proj, height=proj.max() * 0.10, distance=5)

    noise_floor = w * 0.015
    def _zone_gap(zx0, zx1):
        zp = sorted(p for p in peaks if zx0 <= p <= zx1)
        if len(zp) < 2: return 0
        gaps = np.diff(zp)
        return int(np.max(gaps))

    gap_A = _zone_gap(int(w * 0.25), int(w * 0.40))
    gap_B = _zone_gap(int(w * 0.45), int(w * 0.55))

    if gap_A > noise_floor and gap_A >= gap_B:
        layout = '3-table'
        tables = split_three_tables(bgr, pad=pad)
    else:
        layout = '2-table'
        tables = split_left_right(bgr, pad=pad)

    print(f'   Layout detected: {layout}  (gap_A={gap_A} gap_B={gap_B})')
    return layout, tables

print('Image & grid functions ready (v4.11 + 3-table support).')


Image & grid functions ready (v4.11 + 3-table support).


In [6]:
# ══════════════════════════════════════════════════════════════════
# SECTION C – CELL-BY-CELL OCR  v4.8
# Key changes:
#   • NARROW_COLS (9-13): use NARROW_TARGET_H_PX=160 + NARROW_MAX_SCALE
#   • is_bleed_through: stricter dash filter for numeric cols
#   • _prep_narrow: dedicated pipeline for thin element cols
# ══════════════════════════════════════════════════════════════════

def is_bleed_through(text, conf, col_idx):
    t = str(text).strip()
    # Standalone dash/dot = printed blank marker, not real data
    if col_idx in NUMERIC_COLS and DASH_ONLY.match(t):
        return True
    if conf < BLEED_CONF_THRESHOLD:
        for pattern in BLEED_PATTERNS:
            if re.search(pattern, t):
                return True
    if col_idx == 1:
        if len(t) > 8 or (len(t) <= 2 and conf < 0.4):
            return True
    if col_idx in NUMERIC_COLS:
        if not any(c.isdigit() for c in t) and conf < 0.4:
            return True
    return False

def _scale_up(gray, target_h=CELL_TARGET_H_PX, max_scale=CELL_MAX_SCALE):
    h = gray.shape[0]
    if h < 4: return gray
    scale = min(max_scale, max(1.0, target_h / h))
    if scale <= 1.01: return gray
    return cv2.resize(gray, None, fx=scale, fy=scale,
                      interpolation=cv2.INTER_CUBIC)

def _scale_up_narrow(gray):
    """Extra magnification for narrow element columns (9-13)."""
    h = gray.shape[0]
    if h < 4: return gray
    scale = min(NARROW_MAX_SCALE, max(1.0, NARROW_TARGET_H_PX / h))
    if scale <= 1.01: return gray
    return cv2.resize(gray, None, fx=scale, fy=scale,
                      interpolation=cv2.INTER_LANCZOS4)

def _prep_numeric(gray):
    """CLAHE + Otsu + NlMeans + CLOSE + SHARPEN (best for digits)."""
    g = cv2.createCLAHE(2.0, (4,4)).apply(gray)
    _, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.mean(b) < 127: b = cv2.bitwise_not(b)
    b = cv2.fastNlMeansDenoising(b, h=10)
    b = cv2.morphologyEx(b, cv2.MORPH_CLOSE, np.ones((2,2), np.uint8))
    return cv2.filter2D(b, -1, np.array([[-1,-1,-1],[-1,9,-1],[-1,-1,-1]]))

def _prep_text(gray):
    """CLAHE + adaptive thresh (best for month/mixture text)."""
    g = cv2.createCLAHE(3.0, (4,4)).apply(gray)
    b = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                              cv2.THRESH_BINARY, 21, 10)
    if np.mean(b) < 127: b = cv2.bitwise_not(b)
    return cv2.morphologyEx(b, cv2.MORPH_CLOSE, np.ones((2,2), np.uint8))

def _prep_clahe_only(gray):
    """Gray-level fallback: CLAHE + mild sharpen (no binary)."""
    g = cv2.createCLAHE(2.5, (4,4)).apply(gray)
    return cv2.filter2D(g, -1, np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]))

def _prep_narrow_col(gray):
    """
    Dedicated pipeline for narrow element cols (9-13).
    Uses CLAHE 3.0 + bilateral filter (preserves edges without DILATE)
    + OTSU + CLOSE to keep digit strokes clean without merging.
    """
    g = cv2.createCLAHE(3.0, (4,4)).apply(gray)
    g = cv2.bilateralFilter(g, 5, 75, 75)  # edge-preserving denoise
    _, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.mean(b) < 127: b = cv2.bitwise_not(b)
    b = cv2.morphologyEx(b, cv2.MORPH_CLOSE, np.ones((2,2), np.uint8))
    return cv2.filter2D(b, -1, np.array([[-1,-1,-1],[-1,9,-1],[-1,-1,-1]]))

def crop_cell(bgr, hlines, vlines, row, col):
    H, W = bgr.shape[:2]; s = CELL_BORDER_STRIP
    y1 = min(H-1, int(hlines[row])+s);   y2 = max(0, int(hlines[row+1])-s)
    x1 = min(W-1, int(vlines[col])+s);   x2 = max(0, int(vlines[col+1])-s)
    if y2-y1 < 4 or x2-x1 < 4: return None
    crop = bgr[y1:y2, x1:x2].copy()
    p = CELL_PAD_PX
    return cv2.copyMakeBorder(crop, p,p,p,p, cv2.BORDER_CONSTANT,
                              value=(255,255,255))

def ocr_single_cell(bgr_crop, col_idx):
    """
    v4.8: narrow cols use dedicated pipeline + extra magnification.
    All other cols use original 3-variant approach.
    """
    if bgr_crop is None or bgr_crop.size == 0: return '', 0.0
    gray = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2GRAY)

    # Select magnification
    if col_idx in NARROW_COLS:
        gray = _scale_up_narrow(gray)
    else:
        gray = _scale_up(gray)

    # Build preprocessing variants
    if col_idx in NARROW_COLS:
        v1 = _prep_narrow_col(gray)   # narrow-specific
        v2 = _prep_numeric(gray)       # standard numeric
        v3 = _prep_clahe_only(gray)    # gray fallback
    elif col_idx in NUMERIC_COLS:
        v1 = _prep_numeric(gray)
        v2 = _prep_clahe_only(gray)
        cg = cv2.createCLAHE(2.0,(4,4)).apply(gray)
        _, v3 = cv2.threshold(cg, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        if np.mean(v3) < 127: v3 = cv2.bitwise_not(v3)
    else:
        v1 = _prep_text(gray)
        v2 = _prep_clahe_only(gray)
        cg = cv2.createCLAHE(2.0,(4,4)).apply(gray)
        _, v3 = cv2.threshold(cg, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        if np.mean(v3) < 127: v3 = cv2.bitwise_not(v3)

    best_text, best_conf = '', 0.0
    for v in (v1, v2, v3):
        img   = cv2.cvtColor(v, cv2.COLOR_GRAY2BGR) if v.ndim==2 else v
        items = run_ocr(img)
        good  = [(t,s) for _,t,s in items if s >= OCR_MIN_CONF]
        if not good: continue
        text = ' '.join(t for t,_ in good).strip()
        conf = float(np.mean([s for _,s in good]))
        if conf > best_conf:
            best_conf, best_text = conf, text

    if best_text and is_bleed_through(best_text, best_conf, col_idx):
        return '', 0.0
    return best_text, best_conf

def extract_all_cells(bgr, hlines, vlines):
    nrows = len(hlines)-1; ncols = len(vlines)-1
    print(f'  Cell OCR: {nrows}x{ncols} = {nrows*ncols} cells')
    results = {}; filtered_bleed = 0; t0 = time.time()
    for r in range(nrows):
        for c in range(ncols):
            crop = crop_cell(bgr, hlines, vlines, r, c)
            text, conf = ocr_single_cell(crop, c)
            if text:
                results[(r,c)] = {'text': text, 'conf': conf}
            elif conf == 0.0:
                filtered_bleed += 1
        if (r+1) % 10 == 0:
            print(f'  Row {r+1}/{nrows} | {time.time()-t0:.0f}s')
    print(f'  Done: {len(results)}/{nrows*ncols} | bleed={filtered_bleed} | {time.time()-t0:.1f}s')
    return results, nrows, ncols

print('Cell OCR functions ready (v4.8 narrow-col improvement).')


Cell OCR functions ready (v4.8 narrow-col improvement).


In [7]:
# ══════════════════════════════════════════════════════════════════
# SECTION D – POST-PROCESSING  v4.9  (fuzzy-based cleaning)
# ══════════════════════════════════════════════════════════════════
def clean_numeric(text):
    text = str(text).strip()
    if not text or DASH_ONLY.match(text): return ''
    spaced = re.sub(r'(?<=\d)\s+(?=\d)', '', text)
    nums = re.findall(r'\d+(?:[.,]\d+)?', spaced)
    if nums: return nums[0] if len(nums)==1 else ' '.join(nums)
    nums2 = re.findall(r'\d+(?:[.,]\d+)?', text)
    if nums2: return nums2[0] if len(nums2)==1 else ' '.join(nums2)
    return ''.join(c for c in text if c.isdigit() or c in '.,/-+').strip()

def clean_cell(text, col_idx):
    """Per-column cleaning using fuzzy matcher for month and fert cols."""
    text = str(text).strip()
    if not text: return ''
    if col_idx in NUMERIC_COLS and DASH_ONLY.match(text): return ''
    if col_idx in NUMERIC_COLS: return clean_numeric(text)
    if col_idx == 1: return fuzzy_match_month(text)   # ← fuzzy
    if col_idx == 5: return fuzzy_match_fert(text)    # ← fuzzy
    return ' '.join(text.split())

def post_process_grid(raw_results, nrows, ncols):
    cleaned = {}
    for (r,c), info in raw_results.items():
        val = clean_cell(info['text'], c)
        if val: cleaned[(r,c)] = val
    return cleaned

def detect_data_start(raw_results, nrows, max_scan=15, fallback=5):
    """
    Dual-method dynamic header skip.

    Method 1 - Grid formula (primary):
        skip = nrows - EXPECTED_ROWS_AFTER_SKIP
        Grid 69->4 | Grid 70->5 | Grid 71->6

    Method 2 - OCR content scan (validation):
        Scans rows top-down for col-0=row-number, col-1=month.
        Back-calculates: col-0="2" at row 5 -> actual start = row 4.

    Decision:
        |grid - ocr| <= 1  -> use grid formula  (reliable math)
        |grid - ocr|  > 1  -> use OCR scan      (content wins)
        grid <= 0          -> use OCR scan      (bad detection)
    """
    # Method 1: grid formula
    skip_grid = max(0, nrows - EXPECTED_ROWS_AFTER_SKIP)

    # Method 2: OCR content scan
    NO_PAT    = re.compile(r'^\d{1,2}$')
    MONTH_PAT = re.compile(
        r'^(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec'
        r'|Bep|Nnv|Auu|Jau|Fcd|Col|Mav|Tun|Sct|Aot|Nec|Fob|Map|Seo'
        r'|PR|PRN)',
        re.IGNORECASE)

    skip_ocr  = None
    for r in range(min(nrows, max_scan)):
        t0 = str(raw_results.get((r, 0), {}).get('text', '')).strip()
        t1 = str(raw_results.get((r, 1), {}).get('text', '')).strip()
        has_no    = bool(NO_PAT.match(t0))
        has_month = bool(MONTH_PAT.match(t1))
        n_filled  = sum(1 for c in range(17)
                        if raw_results.get((r, c), {}).get('text', '').strip())
        triggered = ((has_no and has_month) or
                     (has_no and n_filled >= 3) or
                     (has_month and n_filled >= 4))
        if triggered:
            row_num  = int(t0) if (has_no and t0.isdigit()) else 1
            skip_ocr = max(0, r - (row_num - 1))
            break

    if skip_ocr is None:
        skip_ocr  = fallback
        ocr_label = f'fallback({fallback})'
    else:
        ocr_label = str(skip_ocr)

    # Decision
    if skip_grid <= 0:
        skip, reason = skip_ocr, 'grid<=0, using OCR'
    elif abs(skip_grid - skip_ocr) <= 1:
        skip, reason = skip_grid, 'methods agree'
    else:
        skip, reason = skip_ocr, f'mismatch(grid={skip_grid},ocr={ocr_label}), using OCR'

    print(f'    [DynSkip] nrows={nrows} grid={skip_grid} '
          f'ocr={ocr_label} -> skip={skip} ({reason})')
    return skip


def apply_scan_header_skip(cells, conf_dict, skip):
    new_cells = {}; new_conf = {}
    for (r,c), val in cells.items():
        if r < skip: continue
        r2 = r - skip
        new_cells[(r2,c)] = val
        if (r,c) in conf_dict: new_conf[(r2,c)] = conf_dict[(r,c)]
    max_r = max((r for r,c in new_cells), default=-1)+1
    print(f'  Skipped {skip} header rows → {max_r} data rows, {len(new_cells)} cells')
    return new_cells, new_conf

def find_data_extent(cells, ncols, max_empty_streak=8):
    if not cells: return 0
    max_r = max(r for r,c in cells)
    last_valid = -1; streak = 0
    for r in range(max_r+1):
        month_v = str(cells.get((r,1),'')).strip()
        no_v    = str(cells.get((r,0),'')).strip()
        has_no    = bool(re.fullmatch(r'\d{1,3}', no_v))
        has_month = bool(re.fullmatch(r'[A-Z][a-z]{1,4}|PR|PRN|TOTAL', month_v))
        is_total  = 'TOTAL' in month_v.upper() or 'TOTAL' in no_v.upper()
        n_cells   = sum(1 for c in range(ncols) if cells.get((r,c),''))
        if (has_no or has_month or is_total) and n_cells >= 1:
            last_valid = r; streak = 0
        else:
            streak += 1
            if streak >= max_empty_streak and last_valid >= 0: break
    return max(last_valid, 0)

def trim_data_region(cells, hlines, vlines,
                     min_row_fill=0.05, min_col_fill=0.05):
    nrows = len(hlines)-1; ncols = len(vlines)-1
    if nrows==0 or ncols==0: return cells, hlines, vlines
    row_cnt = [sum(1 for c in range(ncols) if cells.get((r,c),''))
               for r in range(nrows)]
    col_cnt = [sum(1 for r in range(nrows) if cells.get((r,c),''))
               for c in range(ncols)]
    r_thresh = max(1, int(ncols*min_row_fill))
    c_thresh = max(1, int(nrows*min_col_fill))
    act_r = [i for i,v in enumerate(row_cnt) if v >= r_thresh]
    act_c = [i for i,v in enumerate(col_cnt) if v >= c_thresh]
    if not act_r or not act_c: return cells, hlines, vlines
    r0,r1 = min(act_r),max(act_r)
    c0,c1 = min(act_c),max(act_c)
    new_cells = {(r-r0,c-c0): v for (r,c),v in cells.items()
                 if r0<=r<=r1 and c0<=c<=c1}
    return new_cells, hlines[r0:r1+2], vlines[c0:c1+2]

print('Post-processing functions ready (v4.10 + dynamic header skip).')


Post-processing functions ready (v4.10 + dynamic header skip).


In [8]:
# ══════════════════════════════════════════════════════════════════
# SECTION E – TEMPLATE-BASED EXCEL WRITER  (identical to previous)
# ══════════════════════════════════════════════════════════════════
def write_filled_template(cells_by_table, conf_by_table,
                          col_widths_by_table, row_hts_by_table,
                          template_path, out_path):
    # Dynamic sheet labels: 2-table uses Left/Right, 3-table uses Table_1/2/3
    n_tables = len(cells_by_table)
    if n_tables <= 2:
        side_labels = {1: 'Left_Table', 2: 'Right_Table'}
    else:
        side_labels = {i: f'Table_{i}' for i in cells_by_table}
    out_wb = Workbook(); out_wb.remove(out_wb.active)

    for t_idx, cells in cells_by_table.items():
        side = side_labels.get(t_idx, f'Table_{t_idx}')
        try:
            tmpl_wb = load_workbook(template_path); ws_src = tmpl_wb.active
        except Exception:
            tmpl_wb = Workbook(); ws_src = tmpl_wb.active

        if t_idx in col_widths_by_table:
            for ci,w in enumerate(col_widths_by_table[t_idx]):
                ws_src.column_dimensions[get_column_letter(ci+1)].width = px_to_col_width(w)
        if t_idx in row_hts_by_table:
            for ri,h in enumerate(row_hts_by_table[t_idx]):
                ws_src.row_dimensions[TEMPLATE_HEADER_ROWS+ri+1].height = px_to_row_ht(h)

        cd = conf_by_table.get(t_idx, {})
        for (r,c), val in cells.items():
            excel_row = TEMPLATE_HEADER_ROWS + r + 1
            if excel_row <= TEMPLATE_HEADER_ROWS: continue
            cell = ws_src.cell(row=excel_row, column=c+1, value=val)
            cell.border = BORDER_THIN
            is_total = str(val).upper().strip() == TEMPLATE_TOTAL_LABEL
            if is_total:
                cell.fill = FILL_TOTAL
                cell.font = Font(bold=True)
                cell.alignment = Alignment(horizontal='center', vertical='center')
            else:
                conf_val = cd.get((r,c), 1.0)
                cell.fill = FILL_LOWCONF if conf_val < 0.55 else FILL_DATA
                if c in NUMERIC_COLS:
                    cell.alignment = Alignment(horizontal='right',
                                               vertical='center', wrap_text=True)
                else:
                    cell.alignment = Alignment(horizontal='center',
                                               vertical='center', wrap_text=True)
                if c == 1: cell.font = Font(bold=True)

        ws_dst = out_wb.create_sheet(title=side)
        for row in ws_src.iter_rows():
            for sc in row:
                dc = ws_dst.cell(row=sc.row, column=sc.column, value=sc.value)
                if sc.has_style:
                    dc.font       = copy.copy(sc.font)
                    dc.border     = copy.copy(sc.border)
                    dc.fill       = copy.copy(sc.fill)
                    dc.alignment  = copy.copy(sc.alignment)
                    dc.number_format = sc.number_format
                    dc.protection = copy.copy(sc.protection)
        for k,cd2 in ws_src.column_dimensions.items():
            ws_dst.column_dimensions[k].width = cd2.width
        for k,rd in ws_src.row_dimensions.items():
            ws_dst.row_dimensions[k].height = rd.height
        print(f'  Sheet "{side}": {len(cells)} data cells written.')

    out_wb.save(out_path)
    print(f'  Saved -> {out_path}')

print('Excel writer ready.')


Excel writer ready.


In [24]:
# ══════════════════════════════════════════════════════════════════
# SECTION F – ACCURACY MEASUREMENT  v4.9
# ══════════════════════════════════════════════════════════════════
# USAGE:
#   Run AFTER main pipeline  → uses all_cells from memory
#   Run STANDALONE anytime   → reads saved Excel file from disk
#
# GT files expected at:
#   data/actual/1703_1.xlsx   ← Left  table ground truth
#   data/actual/1703_2.xlsx   ← Right table ground truth
#   (fallback: 1703_1-2.xlsx for left)
# ══════════════════════════════════════════════════════════════════
from difflib import SequenceMatcher

GT_FILES = {
    1: os.path.join(ACTUAL_FOLDER, '1703_1.xlsx'),
    2: os.path.join(ACTUAL_FOLDER, '1703_2.xlsx'),
}
GT_FALLBACKS = {
    1: [os.path.join(ACTUAL_FOLDER, '1703_1-2.xlsx'),
        os.path.join(ACTUAL_FOLDER, f'{base}_1.xlsx')],
    2: [os.path.join(ACTUAL_FOLDER, '1703_2.xlsx'),
        os.path.join(ACTUAL_FOLDER, f'{base}_2.xlsx')],
}
COL_NAMES_ACC = {
    0:'No',      1:'Month',    2:'Yield_Mo', 3:'Todate',  4:'Annual',
    5:'Mixture', 6:'N',        7:'P',        8:'K',       9:'Mg/MOP',
    10:'Zinc',   11:'Urea',    12:"C'cal",   13:"M'ual",  14:'Weeding',
    15:'Rainfall',16:'Remarks'
}

def _sim_ratio(a, b):
    return SequenceMatcher(None, str(a).lower().strip(),
                           str(b).lower().strip()).ratio()

def _norm_val(v):
    s = str(v).strip()
    if s in ('', 'nan', 'None', '-', '.', 'NaN', '--'): return ''
    try:
        f = float(s.replace(',', ''))
        return str(int(f)) if f == int(f) else str(round(f, 2))
    except:
        return s.lower().strip()

def load_gt_file(t_idx):
    primary = GT_FILES.get(t_idx, '')
    candidates = ([primary] if os.path.exists(primary) else []) +                  GT_FALLBACKS.get(t_idx, [])
    for path in candidates:
        if os.path.exists(path):
            print(f'    GT loaded: {path}')
            df = pd.read_excel(path, sheet_name=0, header=None,
                               skiprows=TEMPLATE_HEADER_ROWS)
            return df
    print(f'    GT not found for table {t_idx}. Tried:')
    for p in candidates: print(f'      {p}')
    return None

def compute_accuracy_v48(predicted_cells, t_idx):
    gt_df = load_gt_file(t_idx)
    if gt_df is None: return None

    records = []
    for (r, c), pred_val in predicted_cells.items():
        try:
            gt_raw  = gt_df.iloc[r, c]
            gt_norm = _norm_val(gt_raw)
            if gt_norm == '': continue
            p_norm  = _norm_val(pred_val)
            sc      = _sim_ratio(gt_norm, p_norm)
            records.append({
                'table':        t_idx,
                'row':          r,
                'col':          c,
                'col_name':     COL_NAMES_ACC.get(c, str(c)),
                'ground_truth': str(gt_raw).strip(),
                'predicted':    str(pred_val).strip(),
                'exact':        gt_norm == p_norm,
                'partial':      sc >= 0.5,
                'similarity':   round(sc, 3)
            })
        except (IndexError, KeyError):
            pass

    if not records:
        print(f'    Table {t_idx}: no comparable cells found.')
        return None

    df  = pd.DataFrame(records)
    n   = len(df)
    e   = df['exact'].sum()   / n * 100
    p50 = df['partial'].sum() / n * 100

    print(f"\n  ── Table {t_idx} ({'Left' if t_idx==1 else 'Right'}) Accuracy ──")
    print(f"  GT cells compared : {n}")
    print(f"  Exact  match      : {e:.1f}%")
    print(f"  Partial (≥0.5)    : {p50:.1f}%")
    print(f"\n  {'Col':>4}  {'Name':<12} {'n':>4} {'Exact%':>7} {'Part%':>7}  Bar")
    print(f"  {'-'*54}")
    for col_id in sorted(df['col'].unique()):
        sub  = df[df['col'] == col_id]
        ce   = sub['exact'].sum()   / len(sub) * 100
        cp   = sub['partial'].sum() / len(sub) * 100
        bar  = '█' * int(ce // 10)
        name = COL_NAMES_ACC.get(col_id, str(col_id))
        print(f"  {col_id:>4}  {name:<12} {len(sub):>4} {ce:>7.1f} {cp:>7.1f}  {bar}")
    return df

def load_cells_from_excel(excel_path):
    """Load predicted cells dict from saved Excel output file."""
    wb  = pd.ExcelFile(excel_path)
    smap = {1: 'Left_Table', 2: 'Right_Table'}
    result = {}
    for t_idx, sheet_name in smap.items():
        if sheet_name not in wb.sheet_names:
            print(f'  Sheet {sheet_name} not found in {excel_path}')
            continue
        df = pd.read_excel(excel_path, sheet_name=sheet_name,
                           header=None, skiprows=TEMPLATE_HEADER_ROWS)
        cells = {}
        for r in range(len(df)):
            for c in range(len(df.columns)):
                val = str(df.iloc[r, c]).strip()
                if val and val not in ('nan', 'None', 'NaN'):
                    cells[(r, c)] = val
        print(f'  {sheet_name}: {len(cells)} non-empty cells loaded')
        result[t_idx] = cells
    return result

def run_accuracy(source=None):
    """
    Run accuracy check.

    source can be:
      None        → auto-detect: use all_cells if available, else load Excel
      dict        → use directly as {t_idx: cells_dict}
      str/Path    → path to Excel file to load
    """
    # ── Determine source of predicted cells ───────────────────────
    if source is None:
        try:
            # Try to use all_cells from pipeline run in memory
            _ = all_cells
            cells_to_check = all_cells
            print('[F] Using all_cells from pipeline run ...')
        except NameError:
            # Fall back to loading saved Excel
            if os.path.exists(out_excel):
                print(f'[F] all_cells not found. Loading from: {out_excel}')
                cells_to_check = load_cells_from_excel(out_excel)
            else:
                print(f'[F] ERROR: no all_cells in memory and no Excel at: {out_excel}')
                print(f'    Run the main pipeline first, or provide an Excel path.')
                return None
    elif isinstance(source, dict):
        cells_to_check = source
        print('[F] Using provided cells dict ...')
    else:
        path = str(source)
        print(f'[F] Loading from: {path}')
        cells_to_check = load_cells_from_excel(path)

    # ── Compute accuracy per table ─────────────────────────────────
    acc_frames = []
    for t_idx in sorted(cells_to_check.keys()):
        df_acc = compute_accuracy_v48(cells_to_check[t_idx], t_idx)
        if df_acc is not None:
            acc_frames.append(df_acc)

    if not acc_frames:
        print('\n  No GT files found or no cells matched.')
        print('  Place GT files at:')
        for t, p in GT_FILES.items(): print(f'    Table {t}: {p}')
        return None

    full_acc = pd.concat(acc_frames, ignore_index=True)
    full_acc.to_csv(out_acc, index=False)
    oe = full_acc['exact'].sum()   / len(full_acc) * 100
    op = full_acc['partial'].sum() / len(full_acc) * 100
    print(f"\n  ══ OVERALL ══")
    print(f"  Total cells : {len(full_acc)}")
    print(f"  Exact       : {oe:.1f}%")
    print(f"  Partial≥0.5 : {op:.1f}%")
    print(f"  Saved       : {out_acc}")
    return full_acc

# ── AUTO-RUN ──────────────────────────────────────────────────────
# Works whether run right after main pipeline OR standalone anytime
acc_result = run_accuracy()


[F] Using all_cells from pipeline run ...
    GT loaded: data/actual\1703_1.xlsx

  ── Table 1 (Left) Accuracy ──
  GT cells compared : 247
  Exact  match      : 74.9%
  Partial (≥0.5)    : 93.9%

   Col  Name            n  Exact%   Part%  Bar
  ------------------------------------------------------
     0  No             50    98.0    98.0  █████████
     1  Month          29    65.5    93.1  ██████
     2  Yield_Mo       27    92.6   100.0  █████████
     3  Todate         31    90.3   100.0  █████████
     4  Annual         32    78.1   100.0  ███████
     5  Mixture        10    60.0   100.0  ██████
     6  N               9    66.7   100.0  ██████
     7  P               2     0.0    50.0  
    11  Urea            1   100.0   100.0  ██████████
    12  C'cal           1     0.0   100.0  
    13  M'ual           2     0.0   100.0  
    14  Weeding        14    42.9    57.1  ████
    15  Rainfall       14    78.6   100.0  ███████
    16  Remarks        25    36.0    80.0  ███
    GT 

In [10]:
# ══════════════════════════════════════════════════════════════════
# SECTION G – MAIN PIPELINE  ← RUN THIS CELL
# ══════════════════════════════════════════════════════════════════
def main():
    print('='*62)
    print(f'  Input : {img_path}')
    print(f'  Output: {out_excel}')
    print(f'  v4.9  : fuzzy month+fert matching, narrow-col OCR')
    print('='*62)

    bgr = cv2.imread(img_path)
    if bgr is None: raise FileNotFoundError(img_path)
    print(f'\n[1] Image: {bgr.shape[1]}x{bgr.shape[0]}px')
    bgr = rectify_document(bgr)
    print(f'    Rectified: {bgr.shape[1]}x{bgr.shape[0]}px')

    if SHOW_DEBUG:
        plt.figure(figsize=(14,8))
        plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        plt.title('Rectified'); plt.axis('off'); plt.show()

    layout, tables = auto_split_tables(bgr)
    print(f'\n[2] Layout: {layout} | Tables: {len(tables)}')

    global all_cells, all_confs
    all_cells = {}; all_confs = {}
    col_widths_px = {}; row_hts_px = {}

    for t_idx, t_bgr in tables:
        if layout == '3-table':
            label = f'Table {t_idx}'
        else:
            label = 'Left' if t_idx==1 else 'Right'
        print(f'\n[3] Table {t_idx} ({label}) - {t_bgr.shape[1]}x{t_bgr.shape[0]}px')

        hlines, vlines = detect_grid(t_bgr)
        nrows = len(hlines)-1; ncols = len(vlines)-1
        print(f'    Grid raw: {nrows}x{ncols}')

        vlines = enforce_min_columns(vlines, MIN_COLS_EXPECTED)
        ncols  = len(vlines)-1
        print(f'    Grid enforced: {nrows}x{ncols}')

        if SHOW_DEBUG:
            vis = t_bgr.copy()
            for y in hlines: cv2.line(vis,(0,int(y)),(vis.shape[1],int(y)),(0,255,0),1)
            for x in vlines: cv2.line(vis,(int(x),0),(int(x),vis.shape[0]),(255,0,0),1)
            plt.figure(figsize=(14,10))
            plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
            plt.title(f'Grid {nrows}x{ncols}'); plt.axis('off'); plt.show()

        raw_results, nrows, ncols = extract_all_cells(t_bgr, hlines, vlines)
        conf_dict = {(r,c): info['conf'] for (r,c),info in raw_results.items()}
        cleaned   = post_process_grid(raw_results, nrows, ncols)

        skip = detect_data_start(raw_results, nrows)  # v4.10 dynamic
        cleaned, conf_dict = apply_scan_header_skip(cleaned, conf_dict, skip)

        nc_d   = max((c for r,c in cleaned), default=0)+1
        last   = find_data_extent(cleaned, nc_d, max_empty_streak=8)
        nb4    = len(cleaned)
        cleaned = {k:v for k,v in cleaned.items() if k[0] <= last+2}
        print(f'    Extent: 0-{last}  (trimmed {nb4-len(cleaned)} garbage cells)')

        cleaned, hlines, vlines = trim_data_region(cleaned, hlines, vlines,
                                                    min_row_fill=0.05,
                                                    min_col_fill=0.05)
        nrows = len(hlines)-1; ncols = len(vlines)-1
        print(f'    Final: {nrows}x{ncols}, {len(cleaned)} cells')

        col_widths_px[t_idx] = [int(vlines[i+1]-vlines[i]) for i in range(ncols)]
        row_hts_px[t_idx]    = [int(hlines[i+1]-hlines[i]) for i in range(nrows)]
        all_cells[t_idx]  = cleaned
        all_confs[t_idx]  = conf_dict

    print(f'\n[4] Writing Excel ...')
    write_filled_template(all_cells, all_confs, col_widths_px, row_hts_px,
                          TEMPLATE_PATH, out_excel)

    print('\nDone ✓  (Run Section F cell next for accuracy)')
    return all_cells

results = main()


  Input : data/images\IMG_1703.jpeg
  Output: data/output\IMG_1703_filled.xlsx
  v4.9  : fuzzy month+fert matching, narrow-col OCR

[1] Image: 3177x2936px
    Rectified: 2902x2741px
   Layout detected: 2-table  (gap_A=34 gap_B=36)

[2] Layout: 2-table | Tables: 2

[3] Table 1 (Left) - 1451x2741px
    Grid raw: 70x17
    Grid enforced: 70x17
  Cell OCR: 70x17 = 1190 cells
  Row 10/70 | 436s
  Row 20/70 | 816s
  Row 30/70 | 1166s
  Row 40/70 | 1456s
  Row 50/70 | 1674s
  Row 60/70 | 1886s
  Row 70/70 | 2096s
  Done: 354/1190 | bleed=831 | 2095.8s
    [DynSkip] nrows=70 grid=5 ocr=6 -> skip=5 (methods agree)
  Skipped 5 header rows → 64 data rows, 263 cells
    Extent: 0-63  (trimmed 0 garbage cells)
    Final: 64x17, 263 cells

[3] Table 2 (Right) - 1457x2741px
    Grid raw: 69x17
    Grid enforced: 69x17
  Cell OCR: 69x17 = 1173 cells
  Row 10/69 | 239s
  Row 20/69 | 483s
  Row 30/69 | 725s
  Row 40/69 | 975s
  Row 50/69 | 1218s
  Row 60/69 | 1448s
  Done: 476/1173 | bleed=697 | 1669.1s

In [11]:
# ══════════════════════════════════════════════════════════════════
# SECTION H – PRECISION/F1 SCORE CALCULATOR  (Add as Cell 11)
# ══════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from difflib import SequenceMatcher

def calculate_f1_scores(acc_df):
    """Compute Precision/Recall/F1 from accuracy dataframe."""
    if acc_df is None:
        print("Run accuracy first! (Section F)")
        return
    
    thresholds = [0.3, 0.4, 0.5, 0.6]
    results = []
    
    for thresh in thresholds:
        # TP: partial matches >= threshold
        # Exact matches always count as TP
        df_thresh = acc_df[acc_df['similarity'] >= thresh]
        tp = len(df_thresh)
        
        # FP: non-matches predicted as matches (approx)
        total_predicted = len(acc_df[acc_df['predicted'] != ''])
        fp = total_predicted - tp
        
        # FN: exact matches below threshold (rare)
        exact_below = len(acc_df[(acc_df['exact'] == True) & 
                                (acc_df['similarity'] < thresh)])
        fn = exact_below
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        results.append({
            'Threshold': thresh,
            'TP': tp, 'FP': fp, 'FN': fn,
            'Precision': f'{precision:.1%}',
            'Recall': f'{recall:.1%}',
            'F1': f'{f1:.1%}'
        })
    
    df_results = pd.DataFrame(results)
    print('OCR Pipeline Metrics:')
    print(df_results.to_string(index=False))
    
    # Save detailed report
    out_f1 = out_excel.replace('.xlsx', '_f1_metrics.csv')
    df_results.to_csv(out_f1, index=False)
    print(f'\nDetailed report: {out_f1}')
    
    return df_results

# ── RUN IT ────────────────────────────────────────────────────────
if 'acc_result' in globals() and acc_result is not None:
    f1_table = calculate_f1_scores(acc_result)
else:
    print("Run Section F (accuracy) first, then this cell!")


OCR Pipeline Metrics:
 Threshold  TP  FP  FN Precision Recall    F1
       0.3 525  29   0     94.8% 100.0% 97.3%
       0.4 517  37   0     93.3% 100.0% 96.5%
       0.5 506  48   0     91.3% 100.0% 95.5%
       0.6 480  74   0     86.6% 100.0% 92.8%

Detailed report: data/output\IMG_1703_filled_f1_metrics.csv
